### Stream Customers Data from cloud files to Delta Lake

 1. Read files from cloud storage using DataStreamReader API
 2. Transform the dataframe to add the following columns\
        - file path: Cloud file path\
        - ingestion date: Current Time Stamp
 3. Write the transformed data stream to a Delta lake table

**1. Read files from Cloud storage using DataStreamReader API**

In [0]:
customers_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/gizmobox_sivan/landing/operational_data_volume/customers_autoloader/_schema")     
        .option("cloudFiles.inferColumnTypes", "true")   
        .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")    
        .option("pathGlobFilter","customers_2024*.json")
        .load("/Volumes/gizmobox_sivan/landing/operational_data_volume/customers_autoloader/")
)

**2. Transform the dataframe to add the following columns**
- file path: Cloud file path
- ingestion data: current time stamp

In [0]:
from pyspark.sql.functions import col, current_timestamp

customers_transformed_df = (
    customers_df.withColumn("file_path", col("_metadata.file_path"))
                .withColumn("ingestion_date", current_timestamp())
)

**3. Write the transformed data stream to a Delta lake table**

[structured-streaming-output-mode](https://learn.microsoft.com/en-us/azure/databricks/structured-streaming/output-mode)

[structured-streaming-triggers](https://learn.microsoft.com/en-us/azure/databricks/structured-streaming/triggers)

In [0]:
streaming_query = (
    customers_transformed_df.writeStream
    .format("delta")
    .option("checkpointlocation", "/Volumes/gizmobox_sivan/landing/operational_data_volume/customers_autoloader/_checkpoint_stream")
    .toTable("gizmobox_sivan.bronze.customers_autoloader")
)


In [0]:
streaming_query.stop()

In [0]:
# spark.sql("drop table if exists gizmobox_sivan.bronze.customers_autoloader")

In [0]:
spark.table("gizmobox_sivan.bronze.customers_autoloader").display()